In [2]:
import os, sys

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.common.spark_session import get_spark_session
from src.ml.config import load_ml_config
from src.ml.training_dataset import load_training_dataset
from src.ml.features import (build_feature_sql, fit_feature_pipeline, feature_names,
                             run_feature_pipeline, load_feature_pipeline)

spark = get_spark_session("ml-feature-pipeline")
cfg = load_ml_config()

26/10/01 16:32:11 WARN Utils: Your hostname, Thilinis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.8 instead (on interface en0)
26/10/01 16:32:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 16:32:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/01 16:32:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
# the SQL generated from config/ml.yaml for the first pipeline stage
print(build_feature_sql(cfg))

SELECT *,
       hour(from_utc_timestamp(CAST(hour_ts AS TIMESTAMP), 'America/New_York')) AS hour_of_day,
       (dayofweek(from_utc_timestamp(CAST(hour_ts AS TIMESTAMP), 'America/New_York')) + 5) % 7 AS day_of_week,
       month(from_utc_timestamp(CAST(hour_ts AS TIMESTAMP), 'America/New_York')) AS month,
       CASE WHEN dayofweek(from_utc_timestamp(CAST(hour_ts AS TIMESTAMP), 'America/New_York')) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend,
       CASE WHEN CAST(to_date(from_utc_timestamp(CAST(hour_ts AS TIMESTAMP), 'America/New_York')) AS STRING) IN ('2024-01-01', '2024-01-15', '2024-02-12', '2024-02-15', '2024-02-19', '2024-05-27', '2024-06-19', '2024-07-04', '2024-09-02', '2024-10-14', '2024-11-05', '2024-11-11', '2024-11-28', '2024-12-25') THEN 1 ELSE 0 END AS is_holiday,
       sin(2 * pi() * (hour(from_utc_timestamp(CAST(hour_ts AS TIMESTAMP), 'America/New_York'))) / 24.0) AS hour_of_day_sin,
       cos(2 * pi() * (hour(from_utc_timestamp(CAST(hour_ts AS TIMESTAMP), 'America/New

In [4]:
import json
summary = run_feature_pipeline(spark, cfg)
print(json.dumps(summary, indent=2))

26/10/01 16:32:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

{
  "seconds_fit_transform_write": 7.21,
  "rows_per_split": {
    "train": 301920,
    "validation": 72594,
    "test": 72816
  },
  "feature_vector_size": 263,
  "numeric_features": [
    "hour_of_day_sin",
    "hour_of_day_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "month",
    "is_weekend",
    "is_holiday",
    "weather_temperature_imp",
    "weather_relative_humidity_imp",
    "weather_precipitation_imp",
    "weather_wind_speed_imp",
    "weather_pressure_imp",
    "air_quality_pm25_imp",
    "demand_lag_1h_imp",
    "demand_lag_24h_imp",
    "demand_lag_168h_imp",
    "demand_rolling_mean_24h_imp",
    "weather_precipitation_missing",
    "air_quality_pm25_missing"
  ],
  "categorical_features": [
    "pickup_location_id",
    "pickup_borough",
    "weather_condition_code"
  ],
  "feature_dataset": "data/ml/feature_dataset",
  "feature_pipeline": "models/feature_pipeline"
}


In [5]:
import pandas as pd
model = load_feature_pipeline(cfg)
names = feature_names(model, cfg)
print("feature vector size:", len(names))
print(pd.Series(names).head(30).to_string())

feature vector size: 263
0                   hour_of_day_sin
1                   hour_of_day_cos
2                   day_of_week_sin
3                   day_of_week_cos
4                             month
5                        is_weekend
6                        is_holiday
7           weather_temperature_imp
8     weather_relative_humidity_imp
9         weather_precipitation_imp
10           weather_wind_speed_imp
11             weather_pressure_imp
12             air_quality_pm25_imp
13                demand_lag_1h_imp
14               demand_lag_24h_imp
15              demand_lag_168h_imp
16      demand_rolling_mean_24h_imp
17    weather_precipitation_missing
18         air_quality_pm25_missing
19             pickup_location_id=1
20            pickup_location_id=10
21           pickup_location_id=100
22           pickup_location_id=101
23           pickup_location_id=102
24           pickup_location_id=106
25           pickup_location_id=107
26           pickup_location_id=108
27 

In [6]:
# which attributes were dropped, and what the final dataset looks like
raw = load_training_dataset(spark, cfg)
final = spark.read.parquet(cfg.feature_dataset_path)
print("dropped:", sorted(set(raw.columns) - set(final.columns)))
print("kept   :", final.columns)
final.limit(5).toPandas()

dropped: ['air_quality_pm25', 'air_quality_station_count', 'demand_lag_168h', 'demand_lag_1h', 'demand_lag_24h', 'demand_rolling_mean_24h', 'pickup_borough', 'pickup_zone', 'trip_count', 'weather_condition_code', 'weather_precipitation', 'weather_pressure', 'weather_relative_humidity', 'weather_temperature', 'weather_wind_speed']
kept   : ['pickup_location_id', 'hour_ts', 'label', 'features', 'split']


,pickup_location_id,hour_ts,label,features,split
0,252,2024-01-08 05:00:00,1.0,"(-0.006379852275301949, 1.415625553973223, -4....",train
1,255,2024-01-08 05:00:00,0.0,"(-0.006379852275301949, 1.415625553973223, -4....",train
2,260,2024-01-08 05:00:00,0.0,"(-0.006379852275301949, 1.415625553973223, -4....",train
3,215,2024-01-08 05:00:00,1.0,"(-0.006379852275301949, 1.415625553973223, -4....",train
4,217,2024-01-08 05:00:00,2.0,"(-0.006379852275301949, 1.415625553973223, -4....",train


In [8]:
raw_cols = set(raw.columns)
final_cols = set(final.columns)
names = feature_names(model, cfg)
used_in_features = sorted(
    c for c in raw_cols - final_cols
    if any(n == c or n.startswith(c + "=") or n.startswith(c + "_") for n in names)
)
print("renamed to label      :", cfg.target)
print("encoded into features :", [c for c in used_in_features if c != cfg.target])
print("removed (not used)    :", sorted(raw_cols - final_cols - set(used_in_features) - {cfg.target}))
print("kept as identifiers   :", sorted(final_cols & raw_cols))

renamed to label      : trip_count
encoded into features : ['air_quality_pm25', 'demand_lag_168h', 'demand_lag_1h', 'demand_lag_24h', 'demand_rolling_mean_24h', 'pickup_borough', 'weather_condition_code', 'weather_precipitation', 'weather_pressure', 'weather_relative_humidity', 'weather_temperature', 'weather_wind_speed']
removed (not used)    : ['air_quality_station_count', 'pickup_zone']
kept as identifiers   : ['hour_ts', 'pickup_location_id', 'split']


In [7]:
# effect of the imputation: values before and after for Manhattan (no PM2.5 monitor)
from pyspark.sql import functions as F
tmp = model.stages[0].transform(raw)
(tmp.filter("pickup_borough = 'Manhattan'")
    .select("hour_ts", "air_quality_pm25", "air_quality_pm25_clean", "air_quality_pm25_missing",
            "weather_precipitation", "weather_precipitation_clean", "weather_precipitation_missing")
    .orderBy("hour_ts").limit(8).toPandas())

,hour_ts,air_quality_pm25,air_quality_pm25_clean,air_quality_pm25_missing,weather_precipitation,weather_precipitation_clean,weather_precipitation_missing
0,2024-01-08 05:00:00,NaN,5.408917,1.0,0.0,0.0,0.0
1,2024-01-08 05:00:00,NaN,5.408917,1.0,0.0,0.0,0.0
2,2024-01-08 05:00:00,NaN,5.408917,1.0,0.0,0.0,0.0
3,2024-01-08 05:00:00,NaN,5.408917,1.0,0.0,0.0,0.0
4,2024-01-08 05:00:00,NaN,5.408917,1.0,0.0,0.0,0.0
5,2024-01-08 05:00:00,NaN,5.408917,1.0,0.0,0.0,0.0
6,2024-01-08 05:00:00,NaN,5.408917,1.0,0.0,0.0,0.0
7,2024-01-08 05:00:00,NaN,5.408917,1.0,0.0,0.0,0.0


In [9]:
(tmp.filter("weather_precipitation_missing = 1")
    .select("hour_ts", "weather_precipitation", "weather_precipitation_clean", "weather_precipitation_missing")
    .orderBy("hour_ts").limit(5).toPandas())

,hour_ts,weather_precipitation,weather_precipitation_clean,weather_precipitation_missing
0,2024-01-13 04:00:00,NaN,0.0,1.0
1,2024-01-13 04:00:00,NaN,0.0,1.0
2,2024-01-13 04:00:00,NaN,0.0,1.0
3,2024-01-13 04:00:00,NaN,0.0,1.0
4,2024-01-13 04:00:00,NaN,0.0,1.0


In [10]:
# Check the final generated ML features

import pandas as pd

# Load the fitted feature pipeline and final feature dataset
model = load_feature_pipeline(cfg)
final = spark.read.parquet(cfg.feature_dataset_path)

# Get the feature names in the same order as the feature vector
names = feature_names(model, cfg)

# Take one example row
row = (
    final
    .select(
        cfg.entity_column,
        cfg.time_column,
        "split",
        "label",
        "features"
    )
    .orderBy(cfg.time_column)
    .first()
)

# Convert Spark vector to normal Python values
values = row["features"].toArray().tolist()

print("Zone :", row[cfg.entity_column])
print("Time :", row[cfg.time_column])
print("Split:", row["split"])
print("Label:", row["label"])

print("\nFeature vector size:", len(values))
print("Feature names size :", len(names))

# Show every feature together with its value
feature_values = pd.DataFrame({
    "feature": names,
    "value": values
})

feature_values

Zone : 252
Time : 2024-01-08 05:00:00
Split: train
Label: 1.0

Feature vector size: 263
Feature names size : 263


,feature,value
0,hour_of_day_sin,-6.379852e-03
1,hour_of_day_cos,1.415626e+00
2,day_of_week_sin,-4.452537e-16
3,day_of_week_cos,1.389614e+00
4,month,-1.069397e+00
...,...,...
258,weather_condition_code=16,0.000000e+00
259,weather_condition_code=10,0.000000e+00
260,weather_condition_code=12,0.000000e+00
261,weather_condition_code=13,0.000000e+00


In [ ]:
spark.stop()